# Induction Heads in GPT-2 Small

An educational walkthrough following **Neel Nanda's** TransformerLens tutorial style and the **Anthropic** induction-heads paper (Olsson et al., 2022).

**Induction heads** are attention heads that implement a remarkably simple in-context rule:

> For sequence `... A B ... A ?`, attend from the second `A` back to `B`, then copy `B` to the output. Predicted next token: `B`.

This is one of the simplest known mechanistic explanations of **in-context learning** in transformers. We're going to find these heads in GPT-2 small, visualize them, and prove they're causally responsible for in-context performance via ablation.

---

**Plan**

1. Load GPT-2 small with TransformerLens.
2. Build *repeated random token* sequences so any pattern matching must be in-context, not memorization.
3. Show the per-token loss drops sharply on the repeated half.
4. Compute an *induction score* for every (layer, head); plot the heatmap.
5. Identify the strongest induction heads.
6. Visualize the diagonal-stripe attention pattern with `circuitsvis`.
7. Zero-ablate the top induction heads — confirm the loss drop disappears.


## 1. Setup

In [ ]:
# Uncomment if needed
# %pip install -q transformer_lens circuitsvis plotly einops

In [ ]:
import torch
import einops
import plotly.express as px
import plotly.graph_objects as go
import circuitsvis as cv
from transformer_lens import HookedTransformer
from IPython.display import display

torch.set_grad_enabled(False)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
print(f"Loaded GPT-2 small: {model.cfg.n_layers} layers x {model.cfg.n_heads} heads = {model.cfg.n_layers * model.cfg.n_heads} attention heads total")
print(f"d_model = {model.cfg.d_model}, d_head = {model.cfg.d_head}, vocab = {model.cfg.d_vocab}")

## 2. The induction-head circuit, conceptually

An induction head is a *2-head circuit* spanning two layers:

- **Previous-token head** (earlier layer): at every position `i`, copies information about token at position `i-1` into the residual stream at position `i`. So position `i` now "knows" what token preceded it.
- **Induction head** (later layer): at the second occurrence of `A`,
  - the **query** reads "I am preceded by `A`'s previous token";
  - the **key** at every position reads "the token before me is X";
  - so Q matches K most strongly at the position right after the *first* occurrence of `A` — i.e., at `B`;
  - the **OV circuit** copies the token at that position (i.e., `B`) into the residual stream;
  - the unembedding then boosts logit for `B`.

This is **K-composition**: the induction head's keys depend on what an earlier head wrote. We won't dissect both heads here — we'll focus on detecting and ablating the induction heads themselves.

## 3. Repeated random token sequences

To prove a head is doing real *in-context* induction (not memorization or token statistics), we feed the model sequences of the form

```
[BOS] r_0 r_1 ... r_{n-1} r_0 r_1 ... r_{n-1}
```

where `r_i` are uniformly random token IDs. The model has never seen these random sequences during training, so any drop in loss on the second half must come from in-context pattern-matching.

In [ ]:
torch.manual_seed(42)
seq_len = 50
batch = 4

# Sample random tokens, avoiding the very low and very high vocab ranges (special/rare tokens)
random_tokens = torch.randint(1000, model.cfg.d_vocab - 1000, (batch, seq_len), device=device)
bos = torch.full((batch, 1), model.tokenizer.bos_token_id, device=device, dtype=torch.long)
repeated_tokens = torch.cat([bos, random_tokens, random_tokens], dim=1)

n_pos = repeated_tokens.shape[1]
print(f"Input shape: {tuple(repeated_tokens.shape)}  # [batch, 1 + 2*seq_len]")
print(f"BOS at pos 0; first copy at pos 1..{seq_len}; second copy at pos {seq_len+1}..{2*seq_len}")

## 4. Per-token loss on the repeated sequence

In [ ]:
logits, cache = model.run_with_cache(repeated_tokens, return_type="logits")
log_probs = logits.log_softmax(dim=-1)

# Loss at position i is the negative log prob of token at position i+1
targets = repeated_tokens[:, 1:]
preds = log_probs[:, :-1]
per_token_loss = -preds.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # [batch, n_pos-1]
mean_loss = per_token_loss.mean(0)

first_half_loss = mean_loss[:seq_len].mean().item()
second_half_loss = mean_loss[seq_len:].mean().item()
print(f"Mean loss, first-half  positions: {first_half_loss:.3f}")
print(f"Mean loss, second-half positions: {second_half_loss:.3f}")
print(f"Drop                                : {first_half_loss - second_half_loss:.3f}")

In [ ]:
fig = px.line(
    y=mean_loss.cpu().numpy(),
    labels={"x": "Position", "y": "Loss"},
    title="Per-token loss on a repeated random sequence",
)
fig.add_vline(x=seq_len, line_dash="dash", line_color="red",
              annotation_text="repeat starts", annotation_position="top")
fig.show()

You should see loss in the first half hover near `~10` (random tokens are unpredictable from prefix alone), and drop sharply at position `seq_len + 1`. That drop is in-context learning happening live — and induction heads are responsible.

## 5. The induction score per head

For a query at position `q` in the *second* copy, the token at `q` equals the token at `q - seq_len` in the first copy. The token *after* `q` in the second copy equals the token at `q - seq_len + 1` in the first copy. So an induction head should attend from `q` to key position `q - seq_len + 1`.

For each (layer, head), we compute the mean attention weight on this shifted diagonal, averaged over second-half query positions. Heads with high scores are induction heads.

In [ ]:
n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads

induction_scores = torch.zeros(n_layers, n_heads)

# Query positions in the second copy
q_idx = torch.arange(seq_len + 1, n_pos, device=device)
k_idx = q_idx - seq_len + 1  # what an induction head should attend to

for layer in range(n_layers):
    pattern = cache["pattern", layer]  # [batch, head, q, k]
    # Advanced index: for each (q, k) pair, pick that attention weight
    induction_attn = pattern[:, :, q_idx, k_idx]  # [batch, head, len(q_idx)]
    induction_scores[layer] = induction_attn.mean(dim=(0, -1)).cpu()

print(f"Induction scores shape: {tuple(induction_scores.shape)}")

In [ ]:
fig = px.imshow(
    induction_scores.numpy(),
    labels={"x": "Head", "y": "Layer", "color": "Induction score"},
    title="Induction score per head in GPT-2 small",
    color_continuous_scale="Blues",
    aspect="auto",
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_yaxes(tickmode="linear", dtick=1)
fig.show()

In [ ]:
# Top-5 induction heads
flat = induction_scores.flatten()
topk = flat.topk(5)
print("Top induction heads (layer, head, score):")
top_heads = []
for score, idx in zip(topk.values, topk.indices):
    layer = (idx // n_heads).item()
    head = (idx % n_heads).item()
    top_heads.append((layer, head))
    print(f"  L{layer}H{head}: {score.item():.3f}")

You should see two heads stand out in layer 5 — typically **L5H1** and **L5H5** — with induction scores well above the rest. Those are GPT-2 small's induction heads.

## 6. Visualizing the attention pattern

In [ ]:
# Visualize the strongest induction head's attention
top_layer, top_head = top_heads[0]
print(f"Visualizing L{top_layer}H{top_head}")

tokens_str = model.to_str_tokens(repeated_tokens[0])
attn_for_head = cache["pattern", top_layer][0, top_head:top_head+1]  # [1, q, k]

display(cv.attention.attention_patterns(
    tokens=tokens_str,
    attention=attn_for_head,
    attention_head_names=[f"L{top_layer}H{top_head}"],
))

You should see a clear **diagonal stripe** in the second half: each query position attends back to a position one step ahead of where its current token last appeared. That's the QK circuit of an induction head, made visible.

## 7. Ablation: are these heads *causally* responsible?

We zero-ablate the top induction heads' output (set `z` to zero, so the head writes nothing into the residual stream) and re-run the model. If these heads are responsible for the in-context loss drop, ablating them should make the second-half loss spike.

In [ ]:
heads_to_ablate = top_heads[:2]  # the two strongest
print(f"Ablating: {heads_to_ablate}")

# Group heads by layer for hook registration
by_layer = {}
for l, h in heads_to_ablate:
    by_layer.setdefault(l, []).append(h)

def make_ablate_hook(head_indices):
    def hook(value, hook):
        # value: [batch, pos, head, d_head]
        value[:, :, head_indices, :] = 0.0
        return value
    return hook

fwd_hooks = [
    (f"blocks.{layer}.attn.hook_z", make_ablate_hook(heads))
    for layer, heads in by_layer.items()
]

ablated_logits = model.run_with_hooks(repeated_tokens, fwd_hooks=fwd_hooks)
ablated_log_probs = ablated_logits.log_softmax(dim=-1)
ablated_loss = -ablated_log_probs[:, :-1].gather(
    -1, repeated_tokens[:, 1:].unsqueeze(-1)
).squeeze(-1).mean(0)

print(f"Original   second-half loss: {mean_loss[seq_len:].mean().item():.3f}")
print(f"Ablated    second-half loss: {ablated_loss[seq_len:].mean().item():.3f}")

In [ ]:
fig = go.Figure()
fig.add_scatter(y=mean_loss.cpu().numpy(), name="Original", line=dict(color="steelblue"))
fig.add_scatter(y=ablated_loss.cpu().numpy(), name="Induction heads ablated", line=dict(color="crimson"))
fig.add_vline(x=seq_len, line_dash="dash", line_color="gray",
              annotation_text="repeat starts", annotation_position="top")
fig.update_layout(
    title="Effect of zero-ablating the top induction heads",
    xaxis_title="Position", yaxis_title="Loss",
)
fig.show()

The ablated curve should stay roughly flat across the second half — the model has lost most of its in-context capability for this task. That's the causal evidence: those two heads do most of the work for repeated-sequence prediction in GPT-2 small.

## 8. Summary

- **What:** Induction heads implement `[A][B] ... [A] → [B]` in-context pattern matching.
- **Detection:** Induction score = mean attention to position `q - seq_len + 1` on a repeated random sequence.
- **In GPT-2 small:** Two heads in layer 5 (typically L5H1 and L5H5) dominate the induction score.
- **Causal evidence:** Zero-ablating these heads destroys most of the in-context loss drop on repeated tokens.
- **Mechanism:** A 2-head circuit — a previous-token head writes "what came before me" into the residual stream; the induction head's QK circuit composes with that signal (K-composition) to attend correctly, while its OV circuit copies the attended-to token to the output.

## Further reading

- Olsson et al., **"In-context Learning and Induction Heads"** (Anthropic, 2022) — the original paper.
- Neel Nanda, **"A Mathematical Framework for Transformer Circuits"** explainers and the **TransformerLens** repo.
- Anthropic's **"A Mathematical Framework for Transformer Circuits"** (Elhage et al., 2021) — the QK/OV decomposition this analysis rests on.
